# Feature Circuits Phase D: GDN Sub-Block SAE

Phase C found microcircuits in MoE sub-block outputs, but all of them encode **task format metadata** (chat boundaries, option letters, task mode switches) — not reasoning.

**Hypothesis**: reasoning substrate in Qwen3.6 lives in the **GDN sub-block** (attention-like output at GDN layers L17/L23), not in MoE sub-block.

**Test**: train SAEs on `layer.self_attn` output at L11 (Gated Attention), L17 (GDN), L23 (GDN). Run same Phase B+C protocol. Compare edges semantically.

- If GDN sub-block edges encode math/logic/arithmetic tokens → reasoning lives in GDN
- If GDN sub-block edges also encode task format → reasoning is truly diffuse in this arch

**Budget**: ~1.5h on RTX 6000.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception: ok = False
if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from safetensors import safe_open
from huggingface_hub import snapshot_download, HfApi, create_repo
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/fc_phase_d'); OUT.mkdir(exist_ok=True)
HF_OUT = 'caiovicentino1/qwen36-feature-circuits'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'

CAPTURE_LAYERS = [11, 17, 23]
D_MODEL = 2048
N_FEATURES = 4096
TOPK = 32
SAE_EPOCHS = 25
SAE_BS = 4096
SAE_LR = 3e-4
MAX_PROMPT_LEN = 1536
N_PROMPTS = 200
MAX_TOKENS_PER_LAYER = 300_000

print('setup done')

## Cell 2 — Load model + inspect GDN structure

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

# Inspect: L11 (GA) vs L17 (GDN)
for L in [11, 17, 23]:
    layer = get_layer_module(model, L)
    print(f'\n=== Layer {L}: {type(layer).__name__} ===')
    for n, m in layer.named_children():
        print(f'  {n}: {type(m).__name__}')
    sa = layer.self_attn
    print(f'  self_attn submodules:')
    for n, _ in sa.named_children():
        print(f'    {n}')

## Cell 3 — Hook self_attn outputs (sub-block delta) + capture

In [ ]:
attn_modules = {L: get_layer_module(model, L).self_attn for L in CAPTURE_LAYERS}

captured_attn = {L: None for L in CAPTURE_LAYERS}

def make_hook(L):
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        captured_attn[L] = h.detach()
    return hook

handles = []
for L in CAPTURE_LAYERS:
    h = attn_modules[L].register_forward_hook(make_hook(L))
    handles.append(h)
print(f'OK {len(handles)} self_attn hooks')

## Cell 4 — Load Stage B prompts + forward + collect activations

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

random.seed(42); random.shuffle(questions)
questions = questions[:N_PROMPTS]
print(f'{len(questions)} prompts')

def format_prompt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = f"Answer the following multiple-choice question step by step.\n\nQuestion: {q}\n\nOptions:\n{choices}\n\nAnswer:"
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

# Forward + collect
attn_acts = {L: [] for L in CAPTURE_LAYERS}
total = {L: 0 for L in CAPTURE_LAYERS}
t0 = time.time()

for q in tqdm(questions, desc='fwd'):
    try:
        p = format_prompt(q['question'], q['options'])
        ids = tok(p, return_tensors='pt').input_ids.cuda()
        if ids.shape[1] > MAX_PROMPT_LEN: continue
        with torch.no_grad():
            _ = model(input_ids=ids, attention_mask=torch.ones_like(ids))
        for L in CAPTURE_LAYERS:
            if captured_attn[L] is None: continue
            a = captured_attn[L][0].float().cpu()
            attn_acts[L].append(a)
            total[L] += a.shape[0]
        if any(total[L] >= MAX_TOKENS_PER_LAYER for L in CAPTURE_LAYERS): break
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue
    except Exception:
        continue

for h in handles: h.remove()
elapsed = (time.time()-t0)/60
print(f'\nfwd done in {elapsed:.1f} min')

acts_attn = {}
for L in CAPTURE_LAYERS:
    X = torch.cat(attn_acts[L], dim=0)
    if X.shape[0] > MAX_TOKENS_PER_LAYER:
        idx = torch.randperm(X.shape[0])[:MAX_TOKENS_PER_LAYER]
        X = X[idx]
    acts_attn[L] = X
    print(f'L{L} attn sub-block: {tuple(X.shape)} mean_norm={X.norm(dim=-1).mean().item():.3f}')

for L in CAPTURE_LAYERS: captured_attn[L] = None
attn_acts = None
torch.cuda.empty_cache()

## Cell 5 — Train TopK SAEs on attn/GDN sub-block outputs

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, d_in, n, k):
        super().__init__()
        self.W_enc = nn.Parameter(torch.empty(d_in, n))
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.W_dec = nn.Parameter(torch.empty(n, d_in))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.k = k
        w = torch.randn(d_in, n) / (d_in ** 0.5)
        self.W_enc.data = w
        self.W_dec.data = w.T.contiguous() / w.T.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        top_v, top_i = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, top_i, F.relu(top_v))
        return z, top_i, top_v
    def decode(self, z): return z @ self.W_dec + self.b_dec
    def forward(self, x):
        z, _, _ = self.encode(x); return self.decode(z), z

def train_sae(X, d_in, n, k, epochs, bs, lr, name=''):
    X = X.cuda()
    sae = TopKSAE(d_in, n, k).cuda()
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    N = X.shape[0]; last_fire = torch.zeros(n, device='cuda'); step = 0
    for e in range(epochs):
        perm = torch.randperm(N, device='cuda')
        el = 0.0; nb = 0
        for i in range(0, N, bs):
            xb = X[perm[i:i+bs]]
            xh, z = sae(xb)
            loss = F.mse_loss(xh, xb)
            opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                sae.W_dec.data /= sae.W_dec.data.norm(dim=-1, keepdim=True).clamp_min(1e-8)
                fired = (z > 0).any(dim=0)
                last_fire[fired] = step
            step += 1; el += loss.item(); nb += 1
        if (e+1) % 5 == 0 or e == 0:
            dead = (step - last_fire) > 500
            n_dead = int(dead.sum().item())
            if n_dead > 0:
                with torch.no_grad():
                    nd = torch.randn(n_dead, d_in, device='cuda')
                    nd /= nd.norm(dim=-1, keepdim=True)
                    sae.W_dec.data[dead] = nd
                    sae.W_enc.data[:, dead] = nd.T
                    sae.b_enc.data[dead] = 0.0
            with torch.no_grad():
                xh, _ = sae(X[:10000])
                var_expl = 1 - ((xh - X[:10000]).var() / X[:10000].var()).item()
            print(f'  {name} ep{e+1:>2}/{epochs} mse={el/nb:.5f} var_expl={var_expl:.3f} dead={n_dead}')
    return sae.cpu()

saes_attn = {}
for L in CAPTURE_LAYERS:
    print(f'\n=== Training attn-SAE on L{L} (n={N_FEATURES}, k={TOPK}) ===')
    sae = train_sae(acts_attn[L], D_MODEL, N_FEATURES, TOPK,
                    SAE_EPOCHS, SAE_BS, SAE_LR, name=f'L{L}_attn')
    saes_attn[L] = sae
    torch.save(sae.state_dict(), OUT / f'sae_attn_L{L}_n{N_FEATURES}_k{TOPK}.pt')
    torch.cuda.empty_cache()

## Cell 6 — Encode + fire→pre corr across layers

In [ ]:
feature_fires = {}
feature_pre = {}
for L in CAPTURE_LAYERS:
    sae = saes_attn[L].cuda()
    X = acts_attn[L].cuda()
    N = X.shape[0]
    fires = torch.zeros(N, N_FEATURES, dtype=torch.bool)
    pre = torch.zeros(N, N_FEATURES, dtype=torch.float16)
    bs = 8192
    with torch.no_grad():
        for i in tqdm(range(0, N, bs), desc=f'enc L{L}'):
            xb = X[i:i+bs]
            z, _, _ = sae.encode(xb)
            p = (xb - sae.b_dec) @ sae.W_enc + sae.b_enc
            fires[i:i+bs] = (z > 0).cpu()
            pre[i:i+bs] = p.to(torch.float16).cpu()
    feature_fires[L] = fires
    feature_pre[L] = pre
    sae.cpu(); torch.cuda.empty_cache()
    fr = fires.float().mean(dim=0)
    print(f'L{L}: healthy(0.5-30%)={((fr>0.005)&(fr<0.30)).sum().item()}/{N_FEATURES}  mean_fire={fr.mean()*100:.2f}%')

healthy_mask = {L: (feature_fires[L].float().mean(0)>0.005) & (feature_fires[L].float().mean(0)<0.30) for L in CAPTURE_LAYERS}

# Fire -> pre-activation correlation
pairs = [(11, 17), (17, 23), (11, 23)]
Tm = 100
summary = {}
for a, b in pairs:
    fires_a = feature_fires[a].float().cuda()
    pre_b = feature_pre[b].float().cuda()
    N = fires_a.shape[0]
    fa = fires_a - fires_a.mean(0); fa = fa / fa.std(0).clamp_min(1e-6)
    pb = pre_b - pre_b.mean(0);     pb = pb / pb.std(0).clamp_min(1e-6)
    corr = (fa.T @ pb) / N

    supp_a = feature_fires[a].float().sum(0)
    supp_b = feature_fires[b].float().sum(0)
    mr = healthy_mask[a] & (supp_a >= Tm)
    mc = healthy_mask[b] & (supp_b >= Tm)
    corr_q = corr[mr][:, mc].cpu()

    summary[f'L{a}->L{b}'] = dict(
        max_abs_corr=float(corr_q.abs().max().item()),
        frac_gt_01=float((corr_q.abs()>0.1).float().mean().item()*100),
        frac_gt_02=float((corr_q.abs()>0.2).float().mean().item()*100))

    print(f'\n=== ATTN/GDN L{a}->L{b} fires->pre corr {tuple(corr_q.shape)} ===')
    print(f'  max |corr|: {summary[f"L{a}->L{b}"]["max_abs_corr"]:.4f}')
    print(f'  frac>0.1: {summary[f"L{a}->L{b}"]["frac_gt_01"]:.3f}%  frac>0.2: {summary[f"L{a}->L{b}"]["frac_gt_02"]:.3f}%')

    rows_idx = mr.nonzero().flatten()
    cols_idx = mc.nonzero().flatten()
    vals, idxs = corr_q.abs().view(-1).topk(10)
    n_cols = corr_q.shape[1]
    print('  top-10:')
    for v, fi in zip(vals.tolist(), idxs.tolist()):
        i = fi // n_cols; j = fi % n_cols
        src = rows_idx[i].item(); dst = cols_idx[j].item()
        print(f'    f{src:>4}->f{dst:>4}  corr={corr_q[i,j].item():+.4f}')
    del fires_a, pre_b, fa, pb, corr; torch.cuda.empty_cache()

print('\n' + '='*60)
print('HEADLINE: attn/GDN vs MoE vs residual max |corr|')
print('  residual (Phase A): 0.009')
print('  MoE (Phase B):      0.71')
for k, v in summary.items(): print(f'  attn/GDN {k}: {v["max_abs_corr"]:.4f}')

## Cell 7 — Semantic characterization of top GDN features

In [ ]:
# Take top 3 source features from each pair's highest corr edges
top_srcs = set()
for a, b in pairs:
    fires_a = feature_fires[a].float().cuda()
    pre_b = feature_pre[b].float().cuda()
    N = fires_a.shape[0]
    fa = fires_a - fires_a.mean(0); fa = fa / fa.std(0).clamp_min(1e-6)
    pb = pre_b - pre_b.mean(0);     pb = pb / pb.std(0).clamp_min(1e-6)
    corr = (fa.T @ pb) / N
    supp_a = feature_fires[a].float().sum(0); supp_b = feature_fires[b].float().sum(0)
    mr = healthy_mask[a] & (supp_a >= Tm); mc = healthy_mask[b] & (supp_b >= Tm)
    corr_q = corr[mr][:, mc].cpu()
    rows_idx = mr.nonzero().flatten()
    vals, idxs = corr_q.abs().view(-1).topk(5)
    n_cols = corr_q.shape[1]
    for v, fi in zip(vals.tolist(), idxs.tolist()):
        src = rows_idx[fi // n_cols].item()
        top_srcs.add((a, src))
    del fires_a, pre_b, fa, pb, corr; torch.cuda.empty_cache()

print(f'Top sources to characterize: {sorted(top_srcs)}')

# Forward 20 prompts and find tokens where each top source fires strongest
handles = [attn_modules[L].register_forward_hook(make_hook(L)) for L in CAPTURE_LAYERS]
random.seed(1)
subset = random.sample(questions, min(20, len(questions)))
hub_fires = {(L, f): [] for (L, f) in top_srcs}

for q in tqdm(subset, desc='char'):
    p = format_prompt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1536: continue
    with torch.no_grad():
        _ = model(input_ids=ids, attention_mask=torch.ones_like(ids))
    tokens_list = [tok.decode([t]) for t in ids[0].tolist()]
    for L in CAPTURE_LAYERS:
        sae = saes_attn[L].cuda()
        pre = ((captured_attn[L][0].float() - sae.b_dec) @ sae.W_enc + sae.b_enc)
        top_v, top_i = pre.topk(TOPK, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, top_i, F.relu(top_v))
        sae.cpu()
        for (layer_t, feat) in top_srcs:
            if layer_t != L: continue
            feat_z = z[:, feat]
            for pos in (feat_z > 0).nonzero().flatten().tolist():
                z_val = float(feat_z[pos])
                ctx_s = max(0, pos-6); ctx_e = min(len(tokens_list), pos+3)
                context = ''.join(tokens_list[ctx_s:ctx_e])
                target = tokens_list[pos]
                hub_fires[(L, feat)].append((z_val, target, context))

for h in handles: h.remove()

print('\n=== ATTN/GDN HUB CHARACTERIZATION ===')
for (L, feat), fires in sorted(hub_fires.items()):
    if not fires: print(f'\nL{L} f{feat}: never fires'); continue
    fires.sort(key=lambda x: -x[0])
    print(f'\nL{L} f{feat}: {len(fires)} fires, top z={fires[0][0]:.2f}')
    for z_val, target, ctx in fires[:6]:
        c = ctx.replace('\n', ' ')[:80]
        print(f'  z={z_val:.2f}  "{target}"  <<  ...{c}...')

## Cell 8 — Upload

In [ ]:
create_repo(HF_OUT, repo_type='model', private=False, exist_ok=True)
api = HfApi()

np.savez(OUT / 'attn_features.npz',
         **{f'L{L}_fires': feature_fires[L].numpy() for L in CAPTURE_LAYERS},
         **{f'L{L}_healthy': healthy_mask[L].numpy() for L in CAPTURE_LAYERS})

with open(OUT / 'summary_corr_attn.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Hub characterizations (first 5 each)
hub_data = {}
for (L, feat), fires in hub_fires.items():
    fires_sorted = sorted(fires, key=lambda x: -x[0])[:10]
    hub_data[f'L{L}_f{feat}'] = [dict(z=v, token=t, context=c) for v, t, c in fires_sorted]
with open(OUT / 'attn_hub_characterization.json', 'w') as f:
    json.dump(hub_data, f, indent=2, ensure_ascii=False)

readme = f'''---
license: mit
base_model: Qwen/Qwen3.6-35B-A3B
tags: [mechanistic-interpretability, gdn, feature-circuits]
---

# Qwen3.6-35B-A3B Feature Circuits — Phase D (Attention/GDN sub-block)

Train SAEs on the output of `layer.self_attn` (= Gated Attention at L11, GDN at L17/L23).
Compare cross-layer correlation magnitude + hub semantics vs Phase B (MoE sub-block).

## Files

- `sae_attn_L{{11,17,23}}_n{N_FEATURES}_k{TOPK}.pt` — attn/GDN sub-block SAEs
- `attn_features.npz` — feature fires
- `summary_corr_attn.json` — cross-layer correlation headline
- `attn_hub_characterization.json` — top activating token contexts per hub

## Comparison

- Residual (Phase A): max corr 0.009
- MoE (Phase B): max corr 0.71 (but input-confounded, ATE ~ 0)
- Attn/GDN (this phase): see summary_corr_attn.json

## Hubs

See attn_hub_characterization.json for semantic interpretation.
If hubs fire on math/logic tokens -> reasoning lives in GDN.
If hubs fire on format tokens -> reasoning is diffuse across all substrates.
'''
with open(OUT / 'README_phase_d.md', 'w') as f: f.write(readme)

for fp in list(OUT.glob('sae_attn_*.pt')) + [OUT/'attn_features.npz',
                                                OUT/'summary_corr_attn.json',
                                                OUT/'attn_hub_characterization.json',
                                                OUT/'README_phase_d.md']:
    try:
        api.upload_file(path_or_fileobj=str(fp), path_in_repo=f'phase_d/{fp.name}',
                        repo_id=HF_OUT, repo_type='model',
                        commit_message='FC Phase D: ' + fp.name)
        print(f'OK {fp.name}')
    except Exception as e:
        print(f'FAIL {fp.name}: {e}')
print(f'\nOK https://huggingface.co/{HF_OUT}/tree/main/phase_d')